# ConvNeXt Five-Model Ensemble Prediction

Adapted from `ConvNeXt-Large_prediction.ipynb`. Load all five cross-validation models and average their softmax probabilities with equal weights. Defaults use epoch-50 EMA weights from `checkpoint_last.pth`, matching the reported cross-validation procedure. Architecture, class ordering, input size, and normalization are read from saved training configurations.

Choose new prediction images in the configuration cell. Predictions on training data are not an independent assessment of ensemble performance.

In [ ]:
from __future__ import annotations

from pathlib import Path
from types import SimpleNamespace
import json
import sys

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from timm import create_model
from tqdm.auto import tqdm


def find_repo_root() -> Path:
    candidate = Path.cwd().resolve()
    while candidate != candidate.parent and not (candidate / "pyproject.toml").exists():
        candidate = candidate.parent
    if not (candidate / "pyproject.toml").exists():
        raise FileNotFoundError("Could not locate the repository root containing pyproject.toml.")
    return candidate

repo_root = find_repo_root()
print(f"Repository root: {repo_root}")

## Configuration

Set `IMAGE_PATH` and `BATCH_INPUT_DIR` to your own images, or use the automatically discovered test images. `CHECKPOINT_NAME = "model_best.pth"` selects best-AUC EMA checkpoints instead. Keep `WEIGHTS_KEY = "model_ema"` for EMA inference.

In [ ]:
RUN_DIR = repo_root / "training/ConvNeXt/5fold_cross_validation"
CHECKPOINT_NAME = "checkpoint_last.pth"
WEIGHTS_KEY = "model_ema"  # Set to "model" only to explicitly use raw weights.
FOLD_NUMBERS = (1, 2, 3, 4, 5)
OUTPUT_DIR = RUN_DIR / "ensemble_prediction_outputs"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
THRESHOLD = 0.5  # Class index 1 is selected only when its probability exceeds this value.
TOP_K_BATCH_ROWS = 10
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

BATCH_DIR_CANDIDATES = [
    repo_root / "training/ConvNeXt/DataSet_Test",
    repo_root / "training/ConvNeXt/test_KO",
    repo_root / "training/ConvNeXt/source_data/LN",
]


def iter_image_paths(root: Path) -> list[Path]:
    return sorted(path for path in root.rglob("*")
                  if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)


BATCH_INPUT_DIR = next((path for path in BATCH_DIR_CANDIDATES
                        if path.is_dir() and next((p for p in path.rglob("*")
                        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES), None) is not None), None)
IMAGE_PATH = next(iter(iter_image_paths(BATCH_INPUT_DIR)), None) if BATCH_INPUT_DIR else None
# Override the discovered paths here, for example:
# BATCH_INPUT_DIR = Path("/path/to/crops")
# IMAGE_PATH = Path("/path/to/example.png")

if BATCH_SIZE < 1 or not 0 <= THRESHOLD <= 1:
    raise ValueError("BATCH_SIZE must be positive and THRESHOLD must lie in [0, 1].")
if IMAGE_PATH is None or not IMAGE_PATH.is_file():
    raise FileNotFoundError("Set IMAGE_PATH to a prediction image in this cell.")
if BATCH_INPUT_DIR is None or not BATCH_INPUT_DIR.is_dir():
    raise FileNotFoundError("Set BATCH_INPUT_DIR to a directory of prediction images in this cell.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Single image: {IMAGE_PATH}")
print(f"Batch root: {BATCH_INPUT_DIR}")

## Load All Five Fold Models

Every fold must have matching architecture, class mapping, and preprocessing. Missing EMA weights raise an error instead of silently substituting raw weights. Models remain in evaluation mode and share one preprocessing pipeline.

In [ ]:
device = torch.device(DEVICE)
models = []
model_metadata = []
training_config = None
config_keys = ("architecture", "class_to_idx", "image_size", "train_mean", "train_std")

for fold_number in FOLD_NUMBERS:
    fold_dir = RUN_DIR / f"fold_{fold_number}"
    checkpoint_path = fold_dir / CHECKPOINT_NAME
    config_path = fold_dir / "config.json"
    if not checkpoint_path.is_file() or not config_path.is_file():
        raise FileNotFoundError(f"Missing checkpoint or config in {fold_dir}")
    config = json.loads(config_path.read_text())
    if any(key not in config for key in config_keys):
        raise ValueError(f"Incomplete training settings: {config_path}")
    if training_config is None:
        training_config = config
    elif any(config[key] != training_config[key] for key in config_keys):
        raise ValueError(f"Fold {fold_number} has incompatible training settings.")
    # Training checkpoints include optimizer state and are trusted local artifacts.
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    checkpoint_config = json.loads(json.dumps(checkpoint.get("config", {})))
    if any(checkpoint_config.get(key) != config[key] for key in config_keys):
        raise ValueError(f"Checkpoint settings disagree with config.json: {checkpoint_path}")
    weights = checkpoint.get(WEIGHTS_KEY)
    if weights is None:
        raise ValueError(f"Missing {WEIGHTS_KEY} weights: {checkpoint_path}")
    mapping = config["class_to_idx"]
    if sorted(mapping.values()) != [0, 1]:
        raise ValueError("Expected exactly two classes with indices 0 and 1.")
    model = create_model(config["architecture"], pretrained=False, num_classes=2)
    model.load_state_dict(weights, strict=True)
    model.to(device).eval()
    models.append(model)
    model_metadata.append({"fold": fold_number, "checkpoint": str(checkpoint_path),
                           "epoch": int(checkpoint["epoch"]), "weights": WEIGHTS_KEY})
    del weights, checkpoint

idx_to_class = {index: label for label, index in training_config["class_to_idx"].items()}
image_size = int(training_config["image_size"])
transform = A.Compose([
    A.LongestMaxSize(max_size=image_size),
    A.PadIfNeeded(image_size, image_size, border_mode=cv2.BORDER_REFLECT_101),
    A.Normalize(mean=tuple(training_config["train_mean"]),
                std=tuple(training_config["train_std"])),
    ToTensorV2(),
])


def read_image_rgb(image_path: Path) -> np.ndarray:
    with Image.open(image_path) as image:
        return np.array(image.convert("RGB"))


@torch.inference_mode()
def predict_images(images: list[np.ndarray]):
    if not images:
        raise ValueError("Provide at least one image.")
    inputs = torch.stack([transform(image=image)["image"].float()
                          for image in images]).to(device)
    # Shape: five folds × batch × two classes. Average probabilities, not logits.
    fold_probs = torch.stack([torch.softmax(model(inputs), dim=1) for model in models])
    ensemble_probs = fold_probs.mean(dim=0).cpu().numpy()
    fold_probs = fold_probs.cpu().numpy()
    predictions = []
    for probabilities in ensemble_probs:
        index = int(probabilities[1] > THRESHOLD)
        predictions.append(SimpleNamespace(
            label=idx_to_class[index], confidence=float(probabilities[index]),
            probabilities={idx_to_class[i]: float(probabilities[i]) for i in range(2)},
        ))
    return predictions, fold_probs

print(f"Architecture: {training_config['architecture']}")
print(f"Image size: {image_size}; device: {device}; classes: {idx_to_class}")
display(pd.DataFrame(model_metadata))

## Single-Image Prediction

This cell runs inference on one crop and visualizes the predicted label together with the class probabilities.


In [ ]:
image_rgb = read_image_rgb(IMAGE_PATH)
predictions, single_fold_probs = predict_images([image_rgb])
prediction = predictions[0]
probability_df = pd.DataFrame(
    {
        "class_name": list(prediction.probabilities.keys()),
        "probability": list(prediction.probabilities.values()),
    }
).sort_values("probability", ascending=False, ignore_index=True)

display(probability_df)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
axes[0].imshow(image_rgb)
axes[0].set_title(f"Prediction: {prediction.label}\nConfidence: {prediction.confidence:.3f}")
axes[0].axis("off")

axes[1].bar(probability_df["class_name"], probability_df["probability"], color=["#546e7a", "#c62828"])
axes[1].set_ylim(0.0, 1.0)
axes[1].set_ylabel("Probability")
axes[1].set_title("Class probabilities",pad=10)
for index, value in enumerate(probability_df["probability"]):
    axes[1].text(index, value + 0.02, f"{value:.3f}", ha="center", va="bottom")
plt.show()

## Export Overlay

This optional step saves a simple labeled overlay for reports or manual review.


In [ ]:
base_image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
text = f"{prediction.label} ({prediction.confidence * 100:.1f}%)"

height, width = base_image.shape[:2]
min_dim = max(1, min(height, width))
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = max(0.3, min(0.7, min_dim / 120.0))
thickness = max(1, int(round(min_dim / 90.0)))
inner_padding = max(4, int(round(min_dim / 18.0)))
outer_padding = max(8, int(round(min_dim / 10.0)))

text_size, baseline = cv2.getTextSize(text, font, font_scale, thickness)
text_width, text_height = text_size
header_height = text_height + baseline + inner_padding * 2
canvas_width = max(width + outer_padding * 2, text_width + outer_padding * 2 + inner_padding * 2)
canvas_height = header_height + height + outer_padding * 2

overlay_bgr = np.full((canvas_height, canvas_width, 3), 255, dtype=np.uint8)
image_x0 = (canvas_width - width) // 2
image_y0 = header_height + outer_padding
overlay_bgr[image_y0:image_y0 + height, image_x0:image_x0 + width] = base_image

header_x0 = (canvas_width - (text_width + inner_padding * 2)) // 2
header_y0 = outer_padding // 2
header_x1 = header_x0 + text_width + inner_padding * 2
header_y1 = header_y0 + header_height
text_org = (header_x0 + inner_padding, header_y1 - baseline - inner_padding)

cv2.rectangle(overlay_bgr, (header_x0, header_y0), (header_x1, header_y1), (0, 0, 0), thickness=-1)
cv2.putText(overlay_bgr, text, text_org, font, font_scale, (0, 255, 255), thickness, cv2.LINE_AA)

overlay_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}_prediction_overlay.png"
cv2.imwrite(str(overlay_path), overlay_bgr)
print(f"Saved overlay to {overlay_path}")


## Batch Prediction

This section walks a directory tree of crop images, performs batched inference, and writes a CSV summary for downstream analysis.


In [ ]:
# BATCH_INPUT_DIR is configured above; it is not overwritten here.
print(f"Predicting images under {BATCH_INPUT_DIR}")

In [ ]:
def infer_source_label(path: Path, root: Path) -> str | None:
    for part in path.relative_to(root).parts[:-1]:
        if part.lower() in {"apoptosis", "pyroptosis", "necroptosis"}:
            return part.lower()
    return None


batch_paths = iter_image_paths(BATCH_INPUT_DIR)
if not batch_paths:
    raise FileNotFoundError(f"No supported images found under {BATCH_INPUT_DIR}")
records = []
for start in tqdm(range(0, len(batch_paths), BATCH_SIZE), desc="Ensemble prediction"):
    paths = batch_paths[start:start + BATCH_SIZE]
    predictions, fold_probs = predict_images([read_image_rgb(path) for path in paths])
    for image_index, (path, pred) in enumerate(zip(paths, predictions)):
        row = {
            "image_path": str(path.resolve()),
            "source_label": infer_source_label(path, BATCH_INPUT_DIR),
            "predicted_label": pred.label,
            "confidence": pred.confidence,
        }
        for class_index, class_name in idx_to_class.items():
            row[f"prob_{class_name}"] = pred.probabilities[class_name]
            row[f"fold_std_{class_name}"] = float(fold_probs[:, image_index, class_index].std(ddof=1))
            for fold_index, fold_number in enumerate(FOLD_NUMBERS):
                row[f"fold_{fold_number}_prob_{class_name}"] = float(fold_probs[fold_index, image_index, class_index])
        records.append(row)

batch_df = pd.DataFrame(records)
batch_csv_path = OUTPUT_DIR / "batch_predictions.csv"
batch_df.to_csv(batch_csv_path, index=False)
metadata = {
    "models": model_metadata, "aggregation": "equal_weight_mean_softmax_probabilities",
    "threshold_class_1": THRESHOLD, "class_to_idx": training_config["class_to_idx"],
    "architecture": training_config["architecture"], "image_size": image_size,
    "mean": training_config["train_mean"], "std": training_config["train_std"],
    "input_directory": str(BATCH_INPUT_DIR.resolve()), "image_count": len(batch_df),
}
(OUTPUT_DIR / "ensemble_prediction_config.json").write_text(json.dumps(metadata, indent=2) + "\n")
summary_df = batch_df.groupby("predicted_label").size().rename("count").reset_index()
display(summary_df)
display(batch_df.head(TOP_K_BATCH_ROWS))
print(f"Saved {len(batch_df)} predictions to {batch_csv_path}")

## If Ground-Truth Labels Are Encoded In Folder Names

For a labeled directory tree, this cell computes a simple agreement table using the class-like folder names found under `BATCH_INPUT_DIR`.


In [ ]:
evaluable_df = batch_df.dropna(subset=["source_label"]).copy()
if evaluable_df.empty:
    print("No class-like folder names were found under the batch input directory; skipping agreement summary.")
else:
    agreement_df = (
        evaluable_df.groupby(["source_label", "predicted_label"])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["source_label", "count"], ascending=[True, False], ignore_index=True)
    )
    evaluable_df["correct_label_match"] = evaluable_df["source_label"] == evaluable_df["predicted_label"]
    accuracy_proxy = float(evaluable_df["correct_label_match"].mean())
    display(agreement_df)
    print(f"Folder-label agreement: {accuracy_proxy:.3f}")
